## Dependencies

In [1]:
using L1DRAC
using CUDA
using LinearAlgebra
using Distributions
using ControlSystemsBase
using StaticArrays
using Plots
using JLD2
using DependencyAtlas
using DifferentialEquations

# System Dynamics

The *true (uncertain) system*:
$$ dX_t = \left[ f(t,X_t) + g(t)U_{\mathcal{L}_1,t} + \Lambda_\mu(t,X_t) \right] dt + \left[p(t,X_t)  \Lambda_\sigma(t,X_t) \right]dW_t, \quad X_0 = x_0 \sim \xi_0. $$
where $X_t \in \mathbb{R}^n$, $U_{\mathcal{L}_1,t} \in \mathbb{R}^{m}$, and  $W_t \in \mathbb{R}^{d}$ is a a standard Brownian motion (SBM).
Additionally, $f(t,X_t) \in \mathbb{R}^n$, $g(t) \in \mathbb{R}^{n \times m}$, and $p(t,X_t) \in \mathbb{R}^{n \times d}$ are **known vector fields,** and $\Lambda_\mu(t,X_t) \in \mathbb{R}^n$ and $\Lambda_\sigma(t,X_t) \in \mathbb{R}^{n \times d}$ are the **drift and diffusion uncertainties**, respectively. 

Dropping the uncertainties, we obtain the *nominal (known) system*:
$$ dX^\star_t = f(t,X^\star_t) dt + p(t,X^\star_t)dW^\star_t, \quad X^\star_0 = x^\star_0 \sim \xi^\star_0, $$
where $\xi^\star_0$ is independent of $\xi^\star_0$ and $W^\star_t$ is an SBM independent of $W_t$

*Remark* Any baseline feedback is absorbed with the known drift $f(t,x)$: Suppose the drift vector field is $f'(t,x)+g(t)U_t$, and we have a baseline law $k(t.\cdot):\mathbb{R}^n \rightarrow \mathbb{R}^m$, then, 
$$f(t,x) \doteq f'(t,x)+g(t)k(t,x), \quad \forall (t,x) \in \mathbb{R}_{\geq 0} \times \mathbb{R}^n$$.

## 1D Double Integrator
The double integrator dynamics is defined by 
$$ 
X_t = \begin{bmatrix} pos(t)  \\ vel(t)  \end{bmatrix}, \quad f(t,x) = (A + BK)x + v(t), \quad g(t) = B,  
$$
where $v(t)$ is a feedfoward signal and $K \in \mathbb{R}^{1 \times 2}$ is a regulator obtained via pole placement, and 
$$ 
A = \begin{bmatrix} 0 & 1 \\ 0 & 0  \end{bmatrix}, \quad  B = \begin{bmatrix} 0  \\ 1  \end{bmatrix}.  
$$
We set $d = 2$, and the remaning vector fields $\Lambda_\mu$, $\Lambda_\sigma$ and, $p$ are set to whatever one wishes to simulate. The initial distributions $\xi_0$ and $\xi_0^star$ are also free to be assigned.

# Setup

## System 

In [2]:
function setup_system(; Ntraj=10) # Ntraj = number of trajectories for ensemble sims, default val 10
    # Simulation Parameters
    tspan = (0.0, 5.0)
    Δₜ = 1e-4 # Time step size
    Δ_saveat = 1e2 * Δₜ # Needs to be an integer multiple of Δₜ
    simulation_parameters = sim_params(tspan, Δₜ, Ntraj, Δ_saveat)

    # System Dimensions
    n, m, d = 2, 1, 2
    system_dimensions = sys_dims(n, m, d)

    # Double integrator dynamics
    A = @SMatrix [0.0 1.0; 0.0 0.0]
    B = @SMatrix [0.0; 1.0]

    # Baseline controller via pole placement
    λ = 10.0 # Stability margin
    sys = ss(A, B, SMatrix{2,2}(1.0I), 0.0)
    K = SMatrix{1,2}(place(sys, -λ * ones(2)))
    dp = (; K) # Dynamics params for GPU

    function baseline_input(t, x, dp) # Tracking controller
        r = @SVector [5*sin(t) + 3*cos(2*t), 0.0] # Reference trajectory
        return dp.K * (r - x)
    end

    # Nominal Vector Fields
    f(t, x, dp) = A * x + B * baseline_input(t, x, dp)
    g(t, x, dp) = @SVector [0.0, 1.0]
    g_perp(t, x, dp) = @SVector [1.0, 0.0]

    p_um(t, x, dp) = 2.0 * @SMatrix [0.01 0.1]
    p_m(t, x, dp) = 1.0 * @SMatrix [0.0 0.8]
    p(t, x, dp) = vcat(p_um(t, x, dp), p_m(t, x, dp))

    nominal_components = nominal_vector_fields(f, g, g_perp, p, dp)

    # Uncertain Vector Fields
    Λμ_um(t, x, dp) = 1e-2 * (1 + sin(x[1]))
    Λμ_m(t, x, dp) = 3.0 * (5 + 10*cos(x[2]) + 5*norm(x))
    Λμ(t, x, dp) = @SVector [Λμ_um(t, x, dp), Λμ_m(t, x, dp)]

    Λσ_um(t, x, dp) = 1e-1 * @SMatrix [0.1+cos(x[2]) 2.0]
    Λσ_m(t, x, dp) = 6 * @SMatrix [0.0 5+sin(x[2])+
                        5.0*(norm(x) < 1 ? norm(x) : sqrt(norm(x)))]
    Λσ(t, x, dp) = vcat(Λσ_um(t, x, dp), Λσ_m(t, x, dp))

    uncertain_components = uncertain_vector_fields(Λμ, Λσ)

    # Initial Distributions
    nominal_ξ₀ = MvNormal(20.0 * ones(2), 1e2 * I(2))
    true_ξ₀ = MvNormal(-2.0 * ones(2), 1e1 * I(2))
    initial_distributions = init_dist(nominal_ξ₀, true_ξ₀)

    # Define Systems
    nominal_system = nom_sys(system_dimensions, nominal_components, 
                        initial_distributions)
    true_system = true_sys(system_dimensions, nominal_components, 
                        uncertain_components, initial_distributions)

    # L1-DRAC Parameters (PLACEHOLDER values)
    ω = 50.0 # Filter bandwidth
    Tₛ = 10 * Δₜ # Sample time (integer multiple of Δₜ)
    λₛ = 100.0 # Predictor stability
    L1params = drac_params(ω, Tₛ, λₛ)

    return (
        simulation_parameters = simulation_parameters,
        nominal_system = nominal_system,
        true_system = true_system,
        L1params = L1params,
        system_dimensions = system_dimensions
    )
end


setup_system (generic function with 1 method)

## main()

Choose any non-empty subset of $\left\{:nominal\_sys, \, :true\_sys, \, :L1\_sys \right\}$ to simulate

In [3]:
function main(; Ntraj = Int(1e1), max_GPUs=10, systems=[:nominal_sys, :true_sys, :L1_sys]) 

    @info "Warmup run for JIT compilation"
    println("=====================================") 
    warmup_setup = setup_system(Ntraj = 10)
    run_simulations(warmup_setup; max_GPUs=max_GPUs, systems=systems);

    println("=====================================")
    @info "Complete run for Ntraj=$Ntraj" 
    println("=====================================")
    setup = setup_system(; Ntraj = Ntraj)
    solutions = run_simulations(setup; max_GPUs=max_GPUs, systems=systems)
    return setup, solutions
end


main (generic function with 1 method)

# Solving

In [5]:
solutions = main(Ntraj=Int(1e3)); # Can also run main(), in which case Ntraj is set to the default chosen in the definition of main(

[ Info: Warmup run for JIT compilation
┌ Warning: Requested 10 GPUs but only 3 available. Using 3.
└ @ L1DRAC /home/adi/.julia/dev/L1DRAC/src/auxiliary.jl:57
[ Info: GPU 0: solving 2 trajectories
[ Info: GPU 1: solving 4 trajectories
[ Info: GPU 2: solving 4 trajectories
┌ Info: Multi-GPU complete
│   num_solutions = 3
│   trajectories_per_gpu =
│    3-element Vector{Int64}:
│     2
│     4
└     4
[ Info: GPU 0: solving 2 trajectories
[ Info: GPU 1: solving 4 trajectories
[ Info: GPU 2: solving 4 trajectories
┌ Info: Multi-GPU complete
│   num_solutions = 3
│   trajectories_per_gpu =
│    3-element Vector{Int64}:
│     2
│     4
└     4
[ Info: GPU 0: solving 2 trajectories
[ Info: GPU 1: solving 4 trajectories
[ Info: GPU 2: solving 4 trajectories
┌ Info: Multi-GPU complete
│   num_solutions = 3
│   trajectories_per_gpu =
│    3-element Vector{Int64}:
│     2
│     4
└     4
[ Info: CPU and GPU memory reclaimed


[ Info: Complete run for Ntraj=1000
┌ Warning: Requested 10 GPUs but only 3 available. Using 3.
└ @ L1DRAC /home/adi/.julia/dev/L1DRAC/src/auxiliary.jl:57
[ Info: GPU 0: solving 200 trajectories
[ Info: GPU 1: solving 400 trajectories
[ Info: GPU 2: solving 400 trajectories
┌ Info: Multi-GPU complete
│   num_solutions = 3
│   trajectories_per_gpu =
│    3-element Vector{Int64}:
│     200
│     400
└     400
[ Info: GPU 0: solving 200 trajectories
[ Info: GPU 1: solving 400 trajectories
[ Info: GPU 2: solving 400 trajectories
┌ Info: Multi-GPU complete
│   num_solutions = 3
│   trajectories_per_gpu =
│    3-element Vector{Int64}:
│     200
│     400
└     400
[ Info: GPU 0: solving 200 trajectories
[ Info: GPU 1: solving 400 trajectories
[ Info: GPU 2: solving 400 trajectories
┌ Info: Multi-GPU complete
│   num_solutions = 3
│   trajectories_per_gpu =
│    3-element Vector{Int64}:
│     200
│     400
└     400
[ Info: CPU and GPU memory reclaimed


# DATA LOGGING

## Saving data

In [8]:
function log_state_results(sols)
    state_logging(solutions; path=joinpath(@__DIR__, "sol_logs"))
end
log_state_results(solutions)

LoadError: FieldError: type Tuple has no field `nominal_sol`, available fields: `1`, `2`

## Loading data

# Plotting

## PLOTS

`idxs` — state selection for `plot(sol)` / `plot(ensemble_sol)`, one grammar, three rules:

**Rule 1**: integers name state components; 0 means time.
- `idxs=1` is X1, `idxs=2` is X2, `idxs=0` is t.

**Rule 2**: a vector means "each of these vs time", overlaid as separate curves.

```julia
plot(sol; idxs=[1,2])   # X1 vs t AND X2 vs t (the default plots all states)
plot(sol; idxs=2)       # just X2 vs t
```

**Rule 3**: a tuple means "pair these as axes" (parametric plot, time implicit).

```julia
plot(sol; idxs=(1,2))   # phase plot: X1 on x-axis, X2 on y-axis
plot(sol; idxs=(0,1))   # t vs X1 — explicit form of the default
plot(sol; idxs=(1,2,3)) # 3D phase trajectory (n >= 3 systems)
```

Summary: list = overlay vs time, tuple = plot against each other.
Plots sample the continuous interpolant, not just the saveat points.
For this double integrator: `idxs=(1,2)` is the position-velocity phase portrait, one curve per trajectory.

In [ ]:
# ensemble_stats: threaded per-timepoint statistics over the FULL ensemble.
# Threads.@threads over timepoints (independent work, needs julia -t N > 1);
# each iteration owns its buffer, sorts once per component, and reads the δ,
# 0.5 (= median exactly), and 1-δ quantiles off the sorted values — so all three
# quantile curves cost a single sort. k-th moment: (E[|Xi|^k])^(1/k), as in the plots.
# Arguments: esol (one EnsembleSolution); k (moment order); δ (quantile level);
#            components (state components to process, default: all)
# Returns: (; t, moment, med, qlow, qhigh) — each stat a Matrix [component, timepoint]
function ensemble_stats(esol; k=2, δ=0.05, components=eachindex(esol.u[1].u[1]))
    t = esol.u[1].t
    Ntraj = length(esol.u)
    nc, nt = length(components), length(t)
    moment = Matrix{Float64}(undef, nc, nt)
    med    = Matrix{Float64}(undef, nc, nt)
    qlow   = Matrix{Float64}(undef, nc, nt)
    qhigh  = Matrix{Float64}(undef, nc, nt)
    Threads.@threads for j in 1:nt
        vals = Vector{Float64}(undef, Ntraj)   # owned by this iteration — no sharing
        for (ci, i) in enumerate(components)
            for tr in 1:Ntraj
                vals[tr] = esol.u[tr].u[j][i]
            end
            moment[ci, j] = (sum(v -> abs(v)^k, vals) / Ntraj)^(1 / k)
            sort!(vals)
            qlow[ci, j]  = quantile(vals, δ;     sorted=true)
            med[ci, j]   = quantile(vals, 0.5;   sorted=true)
            qhigh[ci, j] = quantile(vals, 1 - δ; sorted=true)
        end
    end
    return (; t, moment, med, qlow, qhigh)
end

In [ ]:
# plot_states_vs_time: 3x2 figure (columns = X1 | X2), all three systems overlaid
# per panel (nominal=13, true=7, L1=9):
#   row 1 — sample paths (states vs t), first min(max_traj_plot, Ntraj) paths only
#   row 2 — empirical k-th moments per component: (E[|Xi|^k])^(1/k), the L_k norm
#   row 3 — quantiles (median + [δ, 1-δ] band)
# Rows 2-3 statistics: ONE threaded ensemble_stats pass per system, all trajectories.
# Saved next to this script.
# Arguments: sol (solutions NamedTuple from main, e.g. sols); k (moment order, k=2 -> RMS);
#            δ (probability level: band spans the δ and 1-δ quantiles);
#            max_traj_plot (cap on sample paths drawn in row 1; rows 2-3 always use all);
#            fname (output file name)
# Returns: the figure
function plot_states_vs_time(sol; k=2, δ=0.05, max_traj_plot=500, fname="states_vs_time.png")
    
    # row 1 draws only the first min(max_traj_plot, Ntraj) sample paths
    paths(esol) = 1:min(max_traj_plot, length(esol.u))

    p1 = plot(sol.nominal_sol; idxs=1, trajectories=paths(sol.nominal_sol), color=13,
        linewidth=0.5, linealpha=0.3, title="X1")
    plot!(p1, sol.true_sol; idxs=1, trajectories=paths(sol.true_sol), color=7,
        linewidth=0.5, linealpha=0.3)
    plot!(p1, sol.L1_sol; idxs=1, trajectories=paths(sol.L1_sol), color=9,
        linewidth=0.5, linealpha=0.3)
    
    p2 = plot(sol.nominal_sol; idxs=2, trajectories=paths(sol.nominal_sol), color=13,
        linewidth=0.5, linealpha=0.3, title="X2")
    plot!(p2, sol.true_sol; idxs=2, trajectories=paths(sol.true_sol), color=7,
        linewidth=0.5, linealpha=0.3)
    plot!(p2, sol.L1_sol; idxs=2, trajectories=paths(sol.L1_sol), color=9,
        linewidth=0.5, linealpha=0.3)
    
    # one threaded ensemble_stats pass per system over ALL trajectories:
    # k-th moments (row 2) + median and [δ, 1-δ] quantiles (row 3), states only
    st_nom = ensemble_stats(sol.nominal_sol; k=k, δ=δ, components=1:2)
    st_tru = ensemble_stats(sol.true_sol;    k=k, δ=δ, components=1:2)
    st_L1  = ensemble_stats(sol.L1_sol;      k=k, δ=δ, components=1:2)

    # row 2 — empirical k-th moments
    p3 = plot(st_nom.t, st_nom.moment[1, :]; color=13, lw=1.5, label=false, ylabel="Moment: $k")
    plot!(p3, st_tru.t, st_tru.moment[1, :]; color=7, lw=1.5, label=false)
    plot!(p3, st_L1.t, st_L1.moment[1, :]; color=9, lw=1.5, label=false)

    p4 = plot(st_nom.t, st_nom.moment[2, :]; color=13, lw=1.5, label=false, ylabel="Moment: $k")
    plot!(p4, st_tru.t, st_tru.moment[2, :]; color=7, lw=1.5, label=false)
    plot!(p4, st_L1.t, st_L1.moment[2, :]; color=9, lw=1.5, label=false)

    # row 3 — median line + [δ, 1-δ] quantile band; ribbon takes DISTANCES from
    # the center line, hence med - qlow and qhigh - med
    band(st, i) = (st.med[i, :] .- st.qlow[i, :], st.qhigh[i, :] .- st.med[i, :])

    p5 = plot(st_nom.t, st_nom.med[1, :]; ribbon=band(st_nom, 1), color=13, lw=1.5,
        fillalpha=0.2, label=false, xlabel="t", ylabel="Quantiles: [$δ, $(1 - δ)]")
    plot!(p5, st_tru.t, st_tru.med[1, :]; ribbon=band(st_tru, 1), color=7, lw=1.5,
        fillalpha=0.2, label=false)
    plot!(p5, st_L1.t, st_L1.med[1, :]; ribbon=band(st_L1, 1), color=9, lw=1.5,
        fillalpha=0.2, label=false)

    p6 = plot(st_nom.t, st_nom.med[2, :]; ribbon=band(st_nom, 2), color=13, lw=1.5,
        fillalpha=0.2, label=false, xlabel="t", ylabel="Quantiles: [$δ, $(1 - δ)]")
    plot!(p6, st_tru.t, st_tru.med[2, :]; ribbon=band(st_tru, 2), color=7, lw=1.5,
        fillalpha=0.2, label=false)
    plot!(p6, st_L1.t, st_L1.med[2, :]; ribbon=band(st_L1, 2), color=9, lw=1.5,
        fillalpha=0.2, label=false)

    fig = plot(p1, p2, p3, p4, p5, p6; layout=(3, 2), size=(900, 1100))
    savefig(fig, joinpath(@__DIR__, fname))
    return fig
end

## MAIN

`main()` was an `include`-artifact wrapper: it exists so that loading the script does not run it. In the notebook there is nothing to wrap — its keyword defaults become the parameters cell below, and its body becomes the driver cells after it, in the same order.

Parameters (`main`'s keyword defaults, same values):

In [ ]:
Ntraj = Int(1e1)
max_GPUs = 10
systems = [:nominal_sys, :true_sys, :L1_sys]

### Warmup run for JIT compilation

In [ ]:
@info "Warmup run for JIT compilation"
println("=====================================") 
warmup_setup = setup_system(; Ntraj = 10)
run_simulations(warmup_setup; max_GPUs=max_GPUs, systems=systems);

### Complete run for `Ntraj`

`setup_system` only builds the parameter structs — it runs no simulation, so this cell is also the one to run when you are taking the *load saved states* route below.

In [ ]:
println("=====================================")
@info "Complete run for Ntraj=$Ntraj" 
println("=====================================")
setup = setup_system(; Ntraj = Ntraj)

In [ ]:
solutions = run_simulations(setup; max_GPUs=max_GPUs, systems=systems)

### Save the states to disk

The runnable call documented above: `log_state_results(sols)`. The driver cell binds the run's output to `solutions`, so that is the argument here.

In [ ]:
log_state_results(solutions)

### ALTERNATIVE ENTRY POINT — load saved states instead of simulating

Run **either** the simulate driver cells above **or** this cell, then continue to the plots.

Running it *after* the simulate route is not additive: it rebinds `solutions` to what was read back from disk, so the plots below then show the reloaded ensemble rather than the live run.

Rebuilds the plotting inputs from files saved by `log_state_results` — the recommended route when plotting only states. The L1 file holds the full extended state; `component=:L1_sys_states` slices to X while loading, so summaries and plots do no work on the predictor/filter/estimate blocks. Produces the same NamedTuple shape as the simulate route, so `plot_states_vs_time` accepts live and loaded inputs interchangeably.

- Arguments: `system_dimensions` (`setup.system_dimensions`); `path` (folder of saved files)
- Returns: `(; nominal_sol, true_sol, L1_sol)`

In [ ]:
path = joinpath(@__DIR__, "sol_logs")
system_dimensions = setup.system_dimensions

nominal_sol = load_ensemble(joinpath(path, "states_nominal.jld2"))
true_sol    = load_ensemble(joinpath(path, "states_true.jld2"))
L1_sol      = load_ensemble(joinpath(path, "states_L1.jld2"); component=:L1_sys_states, system_dimensions=system_dimensions) # Since we only want to plot the sattes, we load only the L1 state and not the full extended state which would lead to computing compute-heavy statistics that are not relevant to the plots. 
solutions = (; nominal_sol, true_sol, L1_sol)

### States vs time

3x2 figure, all three systems overlaid per panel. Rows 2-3 run `ensemble_stats` internally, one threaded pass per system over all trajectories — the kernel needs more than one thread for that to be threaded.

*Notebook check (first live run): this cell also writes `states_vs_time.png` through `joinpath(@__DIR__, fname)` — the script's comment reads "Saved next to this script", which a cell is not. Same `@__DIR__` question as the DATA LOGGING section; a file of that name already exists in `examples/ex1/` and would be overwritten if it resolves there.*

In [ ]:
plot_states_vs_time(solutions)

## TO BE DEPRECATED

The marker from the script, carried over as-is. The cell below — the marker comment, the `include("plotting_utils.jl")` line, and `generate_state_plots` — is preserved verbatim. Its fate is a parked decision; nothing here is deleted, fixed, or modernized.

*Notebook check (first live run): `include("plotting_utils.jl")` is a **relative** include. In a script it resolves against the including file's own folder, so it works whatever folder Julia was launched from. At notebook top level there is no enclosing source path, so it resolves against the kernel's working directory instead — this cell throws `SystemError: opening file ... plotting_utils.jl` unless the kernel was started in `examples/ex1/`. The same working-directory question governs `@__DIR__` inside `generate_state_plots`: it sets both the default `path` it reads from and where `states_plot.png` is written. Code unchanged — flagged only.*

In [ ]:
## TO BE DEPRECATED 
include("plotting_utils.jl")

function generate_state_plots(; path=joinpath(@__DIR__, "sol_logs"), max_traj=500)
    nom = load(joinpath(path, "states_nominal.jld2"))
    tru = load(joinpath(path, "states_true.jld2"))
    L1  = load(joinpath(path, "states_L1.jld2"))

    fig = plot_results(nom, tru, L1; max_traj=max_traj)
    savefig(fig, joinpath(@__DIR__, "states_plot.png"))
    @info "Saved states_plot.png"
    return fig
end